# 1. 上下文增强

在基础 RAG 中，检索器返回的片段往往「看起来相关，但读起来不够用」——命中了关键句，却丢失了前后的支撑信息。这不是检索没找对，而是切块粒度导致上下文被截断了。

上下文增强要做的事情很简单：**在命中之后，把周围的关键信息补回来**。本节在 **同一份 `train_dataset.json` 子集**，既看 **inspect 机制**，也看 **LLM 裁判（0～2 分）** 与 **相对 Baseline 的平均分 / 提分题**（文末对比格会写出 **CSV + JSON** 摘要）。

## 环境准备

本节使用智谱 AI 的 `GLM-4-Flash` 做生成模型，使用本地 `BAAI/bge-small-zh-v1.5` 做 embedding。运行前请确保：

1. 安装依赖：除根目录 `requirements.txt`（含 `langchain`、`langchain-community`、`langchain-core`、`langchain-text-splitters`、`langchain-chroma` 等）外，本节另需 `chromadb`、`pandas`、`modelscope`、`sentence-transformers`、`torch`（embedding 本地推理）。**基线与向量库**用 LangChain；**句窗 / Small-to-Big / AutoMerging 的增强逻辑**仍为手写。
2. 在项目根目录的 `.env` 文件中配置 `ZHIPUAI_API_KEY`
3. 首次运行会自动从 ModelScope 下载本地 embedding 模型到当前目录下的 `./models/`
4. （可选）若智谱 API 易触发限流，可设置环境变量 `LLM_SLEEP_AFTER_SUCCESS`（秒）：在每次**成功**的 LLM 调用后追加休眠；代码默认 `0`（不等待，与未设置环境变量一致）。遇 429 时代码仍会按原逻辑退避。

> **数据说明**：本教程使用与「3. 索引阶段」相同的数据集（南瓜书《机器学习公式详解》），保持教程连贯性。


## 统一实验设置（一次定义，后面复用）

下面这组公共代码不再堆在一个超长单元里，而是按「先准备环境，再搭好基线检索，最后补齐评测」的顺序拆开。你可以把它理解成：**先把整节实验的公共底座搭好，后面每种上下文增强方法只改自己的 `context` 构造方式**。

- 数据：`../3. 索引阶段/data/pumpkin_book.pdf`（南瓜书《机器学习公式详解》）
- 问答：`../3. 索引阶段/data/train_dataset.json`
- **共用测试子集**：`QA_INDICES`（当前 **20** 题）
- 生成模型：`glm-4-flash-250414`
- 向量模型：本地 `BAAI/bge-small-zh-v1.5`
- Baseline：字符块 **256 / overlap 20 / k=4**（常见向量 RAG 设定）；送入生成前按 **`CONTEXT_CHAR_BUDGET`** 截断，与各增强路径总字数上限一致
- 对比：`baseline_df` 与 `sentence_window_df` / `small_to_big_df` / `auto_merging_df`
- 跑题流水线：各路径实现 **`*_context(question)`**（如何构图并截断），经 **`answer_from_context_fn`** 得到 `*_answer`，再用 **`run_shared_eval`** 对同一 `qna_dict` 逐题评测（避免重复写 for / prompt / llm_call）
- 评估：LLM 裁判输出 **0～2 分**（`rag_eval_results`）；优先看**平均分**与**相对 Baseline 的提分题**，不必以「全 2 分」为唯一目标
- **独立上下文**：三种增强的「送入生成」文本**仅来自该方法自身**（句窗拼接 / 父块或 AutoMerge 扩展），再统一经 **`CONTEXT_CHAR_BUDGET`** 截断；**AutoMerging 代码单元依赖上一节 Small-to-Big 已定义的子块索引与 `child_to_parent` 等变量**
- 性能：PDF 经 `get_cleaned_pdf_documents()` **只加载、清洗一次**，Baseline / Sentence Window / Small-to-Big 共用；向量库为 LangChain `Chroma`，**固定三套目录**：`./chroma_db/baseline_{chunk}_{overlap}`、`./chroma_db/sentence_window`、`./chroma_db/small_to_big`。改切块参数或换 embedding 时，**删除对应子目录**后重跑即可重建，本节不再维护其它历史目录名。


**基础环境**

公共底座（embedding / PDF / Chroma 工厂 / `llm_call` / `trim_context_to_budget` / `build_rag_generation_prompt` / 题集加载）已经在 `_common.py` 中收纳，前几章也讲过原理，本节直接 `from _common import` 复用。下方仅显式留下 6.1 句窗会用到的 `split_sentences_for_window`。


In [ ]:
import os
import re
import json
import warnings
import pandas as pd
from typing import Callable
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import sys
sys.path.insert(0, ".")
from _common import (
    get_embeddings, get_cleaned_pdf_documents, open_or_build_chroma,
    llm_call, build_rag_generation_prompt, trim_context_to_budget,
    load_qna_subset, CHROMA_COLLECTION, CONTEXT_CHAR_BUDGET,
    PDF_PATH, QA_PATH,
)

# 句级切分仅 6.1 句窗用，留在本节显式可见
def split_sentences_for_window(text: str, max_piece: int = 320):
    raw = [s.strip() for s in re.split(r"(?<=[。！？!?])", text) if s.strip()]
    out = []
    for s in raw:
        if len(s) <= max_piece:
            out.append(s)
        else:
            for i in range(0, len(s), max_piece):
                out.append(s[i : i + max_piece])
    return out

warnings.filterwarnings("ignore")

print("✅ 公共底座（embedding / PDF / Chroma / llm_call / trim）已从 _common 引入")

**本节专用辅助：检索 / 命中解析 / 上下文预览**

下面这些函数是 6.1 baseline / 句窗 / Small-to-Big / AutoMerging 共用的"小工具"——把 retriever 调起来、从命中 Document 里抽 `sentence_id` / `child_id`、按字符上限切定长 chunk、以及预览最终送进 LLM 的上下文。它们的边界都贴 6.1，所以不进 `_common.py`。


In [ ]:
def retriever_hits(retriever, question: str):
    """Runnable 检索接口：invoke 返回 Document 列表。"""
    return retriever.invoke(question)


def hit_sentence_ids_from_docs(docs):
    out = []
    for d in docs:
        sid = d.metadata.get("sentence_id")
        if sid is None:
            print("  [提示] 命中缺少 sentence_id，已跳过。若 persist 目录刚升版，请删旧库后重建索引。")
            continue
        try:
            out.append(int(sid))
        except (TypeError, ValueError):
            print(f"  [提示] 无效 sentence_id={sid!r}，已跳过。")
    seen, uniq = set(), []
    for x in out:
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq


def hit_child_ids_from_docs(docs):
    out = []
    for d in docs:
        cid = d.metadata.get("child_id")
        if cid is None:
            print("  [提示] 命中缺少 child_id，已跳过。若 persist 目录刚升版，请删旧库后重建索引。")
            continue
        try:
            out.append(int(cid))
        except (TypeError, ValueError):
            print(f"  [提示] 无效 child_id={cid!r}，已跳过。")
    seen, uniq = set(), []
    for x in out:
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq


def load_chunks(chunk_size=256, chunk_overlap=20):
    docs = get_cleaned_pdf_documents()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        keep_separator=True,
    )
    return splitter.split_documents(docs)


def build_retriever(chunk_size=256, chunk_overlap=20, k=4):
    persist_dir = f"./chroma_db/baseline_{chunk_size}_{chunk_overlap}"
    chunks = load_chunks(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    ids = [f"b{i}" for i in range(len(chunks))]
    vs = open_or_build_chroma(persist_dir, chunks, ids)
    return vs.as_retriever(search_kwargs={"k": k})


def print_llm_context_preview(final_ctx: str, answer_fn_name: str, mode_label: str) -> None:
    preview = final_ctx[:800] + ("..." if len(final_ctx) > 800 else "")
    print(
        f"\n📎 与 `{answer_fn_name}` 一致：送入 LLM（{mode_label}，budget={CONTEXT_CHAR_BUDGET}）前 800 字：\n{preview}\n"
    )

**再把评测题集固定下来**

为了让 Baseline、句窗、Small-to-Big 和 AutoMerging 的对比尽量公平，这里用 `_common.load_qna_subset` 从 `train_dataset.json` 中按 `QA_INDICES` 取出共用题集。`EVAL_MAX_UNIQ_QUESTIONS` 环境变量可以临时裁剪题集（用于 nbconvert 快速验证）。


In [ ]:
QA_INDICES = list(range(20))
qna_dict = load_qna_subset(QA_PATH, QA_INDICES)

# 测试 / CI 模式：通过环境变量裁剪题集，避免 nbconvert 长时间评估
_lim = os.environ.get("EVAL_MAX_UNIQ_QUESTIONS", "").strip()
if _lim:
    qna_dict = dict(list(qna_dict.items())[: max(0, int(_lim))])
    print(f"⚠️ 测试模式：仅使用前 {len(qna_dict)} 道题（EVAL_MAX_UNIQ_QUESTIONS）")

print(f"✅ 本节共用 {len(qna_dict)} 题（QA_INDICES={QA_INDICES[:5]}...）")

**最后把统一评估流水线补上**

到这里公共底座差最后一块：把"根据 context 生成答案"和"用 LLM 打分"串成一条可复用的流水线。后面每种方法只需要专注**这次要怎么构造 context**；生成、评分、对比表都共用下方 4 段实现。它们也已收纳到 `_common.py`（字符相同），6.2 / 6.3 节直接 `from _common import` 复用。


### 1) 评分 prompt 与 0～2 分标准

LLM-as-Judge 偏严格：仅看「问题 / 参考答案 / 模型答案」三段文字，不读原始 RAG 上下文。0 分是"关键结论错或漏答核心"；1 分是"方向对但缺关键点"；2 分是"要点齐全无明显错误"。判分输出强制为单字符 `0`/`1`/`2`，便于稳定解析。


In [ ]:
DEFAULT_EVAL_PROMPT_2PT = (
    "请作为一名判卷人，按 0～2 分评判「模型答案」回答「用户问题」的质量，并对照「参考答案」核对事实与要点。\n"
    "你只收到下列三段文字，没有 RAG 检索原文；请勿以「未逐字引用原文」为由扣分，除非答案与参考答案明显矛盾。\n"
    "评分标准：\n"
    "2 分：核心结论正确，问题所问的必答要点基本齐全，无明显事实错误；表述可比参考答案更短。\n"
    "1 分：方向基本正确，但明显缺少部分关键点，或解释不完整、表述含糊；无严重编造。\n"
    "0 分：关键结论错误、严重漏答问题核心、或与参考答案明显矛盾、或凭空补充。\n"
    "参考答案中的引导语、举例、排版说明不必复述；勿因未覆盖参考答案中的次要枝节就将 2 分打成 1 分。\n\n"
    "用户问题：{question}\n"
    "参考答案：{expected_answer}\n"
    "模型答案：{llm_answer}\n\n"
    "请仅输出一行，且该行只包含一个字符：0、1 或 2，不要输出任何其它文字。"
)


def _parse_eval_score(raw: str) -> int:
    text = (raw or "").strip()
    for line in text.splitlines():
        s = line.strip()
        s = re.sub(r"^[-*•\d.)]+\s*", "", s)
        if re.fullmatch(r"[012]", s):
            return int(s)
    m = re.search(r"(?<![0-9])([012])(?![0-9])", text)
    if m:
        return int(m.group(1))
    return 0


def simple_eval_2pt(
    llm_answer: str,
    expected_answer: str,
    question: str = "",
    *,
    prompt_template: str | None = None,
) -> int:
    """0~2 分 LLM 裁判。三节都用同一接口；prompt_template 允许各节传入定制模板。
    模板必须包含 {question} / {expected_answer} / {llm_answer} 三个占位符。
    """
    template = prompt_template or DEFAULT_EVAL_PROMPT_2PT
    prompt = template.format(
        question=question,
        expected_answer=expected_answer,
        llm_answer=llm_answer,
    )
    try:
        return _parse_eval_score(llm_call(prompt))
    except Exception as e:
        print(f"评估失败: {e}")
        return 0

> 这个函数已收纳到 `_common.py`（字符相同），6.2 / 6.3 节会直接 `from _common import simple_eval_2pt` 复用，不再重复展示。


### 2) 钩子胶水 `answer_from_context_fn`

本节每个方法只暴露一个钩子 `*_context(question) -> str`（只负责构造上下文）。`answer_from_context_fn` 把"上下文 → 生成答案"这一步统一封装：拼 RAG prompt、调 `llm_call`，输出最终答案字符串。下一节起 `pipeline.run` / `system.ask` 已经直接产出答案，就不再需要它。


In [ ]:
def answer_from_context_fn(build_context_fn: Callable[[str], str]) -> Callable[[str], str]:
    """6.1 专用胶水：把 *_context(q)->str 的钩子接到 LLM 上。
    6.2 / 6.3 不需要这个胶水（pipeline / system.ask 已经直接产出答案）。
    """
    def _answer(question: str) -> str:
        context = build_context_fn(question)
        prompt = build_rag_generation_prompt(question, context)
        return llm_call(prompt)
    return _answer

> 同样收纳到 `_common.py`；本节后续与下一节都通过 `import` 使用。


### 3) 通用评测流水线 `run_shared_eval`

输入是一个 `answer_fn(question) -> str` 和题集 `qna_dict`，逐题调答 → 喂给 `simple_eval_2pt` 打分 → 汇成一张 4 列 DataFrame（`question / llm_answer / expected_answer / rag_eval_results`）。本节四种方法、6.2 节的 `*_pipeline` 都满足同一形状，所以可共用同一函数。


In [ ]:
def run_shared_eval(
    answer_fn: Callable[[str], str],
    qna_dict: dict[str, str],
    *,
    eval_prompt_template: str | None = None,
) -> pd.DataFrame:
    """通用评测流水线：逐题调 answer_fn，再用 simple_eval_2pt 打分，返回 DataFrame。
    6.1 / 6.2 都用本函数；6.3 用 run_session_eval。
    eval_prompt_template 用于 6.2 维度计分等需要定制 prompt 的场景。
    """
    rows = []
    for question, expected in qna_dict.items():
        answer = answer_fn(question)
        score = simple_eval_2pt(
            answer, expected, question,
            prompt_template=eval_prompt_template,
        )
        rows.append({
            "question": question,
            "llm_answer": answer,
            "expected_answer": expected,
            "rag_eval_results": score,
        })
    return pd.DataFrame(rows)

> 同样收纳到 `_common.py`。6.2 节的 `*_pipeline` 也是 `q -> str` 的形状，所以可以直接喂给 `run_shared_eval`。


## Baseline：纯向量检索先跑一遍

先不做任何上下文增强，只用基础向量检索回答 `qna_dict`。

失败观察重点：
- 命中内容是否相关；
- 最终回答是否相关但不完整。

In [ ]:
# Baseline：固定长度分块 + 向量 top-k（与 `load_chunks` 默认尺度一致，贴近常见入门 RAG）
baseline_retriever = build_retriever(chunk_size=256, chunk_overlap=20, k=4)


def baseline_hits(question: str):
    return retriever_hits(baseline_retriever, question)


def baseline_expanded_join(docs) -> str:
    """top-k 块按检索顺序拼接（未做 CONTEXT_CHAR_BUDGET）。"""
    return "\n\n".join(d.page_content for d in docs)


def baseline_context_from_docs(docs) -> str:
    return trim_context_to_budget(baseline_expanded_join(docs), CONTEXT_CHAR_BUDGET)


def baseline_context(question: str) -> str:
    return baseline_context_from_docs(baseline_hits(question))


baseline_answer = answer_from_context_fn(baseline_context)
baseline_df = run_shared_eval(baseline_answer, qna_dict)
baseline_df


### 🔍 结果透视：Baseline 到底检索到了什么？

（**inspect 与同题对比表**怎么读，见下文 **「本节通用读法」**（在 Baseline 失败分析之后）。）

以第 0 个问题为例，查看 Baseline 的 **top-k** 原始片段（本实验 **k=4**；逐条打印**未**做 `CONTEXT_CHAR_BUDGET` 截断）。**单元格末尾**会打印与 `baseline_answer` 一致的「送入 LLM 前 800 字」预览（来自同一次检索的 `baseline_context_from_docs`）。

In [ ]:
# 与 baseline_hits / baseline_expanded_join / baseline_context 同一数据流；片段为未 budget 的原始 top-k
def inspect_baseline(question):
    docs = baseline_hits(question)
    print(f"❓ 问题: {question}\n")
    print(f"📦 检索到 {len(docs)} 个片段:\n")
    for i, doc in enumerate(docs):
        print(f"--- 片段 {i+1} ---\n")
        print(doc.page_content)
    print("\n" + "=" * 50 + "\n")
    print_llm_context_preview(
        baseline_context_from_docs(docs),
        "baseline_answer",
        "定长 chunk",
    )

test_q = list(qna_dict.keys())[0]
inspect_baseline(test_q)

### Baseline 失败分析

Baseline 使用 **256 字块、overlap=20、k=4**，属于很常见的配置；在教材/长叙述 PDF 上仍常出现 **命中块相关但缺前后句、缺同段其它句** 的情况。下面三种方法在 **同一测试集** 上各自用 **独立检索得到的上下文**（句窗 / 父块 / AutoMerge），再与 Baseline 一样经 **`CONTEXT_CHAR_BUDGET`** 截断后生成，便于对照「换策略」带来的差异。**0～2 分**仍受裁判与生成随机性影响，请结合 inspect 一起看。


### 本节通用读法：inspect 与同题对比表

- **不要只看** LLM 最终回答（可能脑补）；更要看各节 **inspect** 里检索/扩展出的正文，以及 **文末同题对比表**。
- **0～2 分**受裁判口径与生成随机性影响；讨论机制时优先看 **相对 Baseline 的提分题**、**平均分**，并对照各节 inspect 末尾「送入 LLM 前 800 字」预览。
- **公平对比**：各增强路径**独立检索**，仅 **`CONTEXT_CHAR_BUDGET`** 在进入生成前统一截断。

## Sentence Window（句子窗口检索）

> 对「无句号且过长」的片段再切段，减轻 PDF 目录被当作一整句、窗口拖入海量无关内容的情况。

> **运行顺序**：句窗索引在本单元内自建；与 Baseline 的对比在文末表格完成。生成用上下文 = **各命中句的窗口拼接**（`k=4` 句命中 → 合并窗口去重），再按 **`CONTEXT_CHAR_BUDGET`** 截断。

回到 baseline 的失败案例：检索器命中了关键句，但紧接着的补充说明落在了下一个 chunk 里。问题不在检索，而在于**命中句的前后支撑句被丢掉了**。

### 核心思想

Sentence Window 的思路非常直觉——既然丢的是前后几句，那就在检索命中后把左右邻居补回来：

1. **索引时**：把文档按句子切分，每个句子单独嵌入，同时在元数据中记录该句子的前后邻居
2. **检索时**：先用句子级 embedding 做检索，命中后不直接把这一句送给 LLM，而是查邻居映射，把左右各 N 句拼上，再送给 LLM

### 与 Baseline 的区别

| 阶段 | Baseline | Sentence Window |
|---|---|---|
| 索引粒度 | 固定字符块 | 句子级 |
| 检索对象 | 整个 chunk | 单个句子 |
| 返回内容 | 命中的 chunk | 命中句子 + 前后窗口 |

窗口固定；证据若分散在不同段落，可配合 Small-to-Big。


In [ ]:
base_docs = get_cleaned_pdf_documents()
# 全书按页顺序拼成一条长文本，再分句；句序即 neighbor_map 中的前后邻关系
full_text = "\n".join(d.page_content for d in base_docs)

sentences = split_sentences_for_window(full_text)
print(f"总句子数: {len(sentences)}")

sentence_map = {i: s for i, s in enumerate(sentences)}
WINDOW_SIZE = 2
# 句 i 的前后各 WINDOW_SIZE 句的全局下标（含 i）
neighbor_map = {
    i: list(range(max(0, i - WINDOW_SIZE), min(len(sentences), i + WINDOW_SIZE + 1)))
    for i in sentence_map
}

persist_dir_sw = "./chroma_db/sentence_window"
sentence_docs = [
    Document(page_content=s, metadata={"sentence_id": str(i)})
    for i, s in sentence_map.items()
]
sentence_vs = open_or_build_chroma(
    persist_dir_sw,
    sentence_docs,
    [str(i) for i in sentence_map.keys()],
)
sentence_retriever = sentence_vs.as_retriever(search_kwargs={"k": 4})


def sentence_window_hit_ids(question: str):
    return hit_sentence_ids_from_docs(retriever_hits(sentence_retriever, question))


def sentence_window_expanded_join(hit_ids) -> str:
    """多命中句的窗口取并集，去重后按句序拼接（未做 CONTEXT_CHAR_BUDGET）。"""
    window_ids = sorted({nid for hid in hit_ids for nid in neighbor_map.get(hid, [hid])})
    return "\n".join(sentence_map[i] for i in window_ids)


def sentence_window_context(question: str) -> str:
    return trim_context_to_budget(
        sentence_window_expanded_join(sentence_window_hit_ids(question)),
        CONTEXT_CHAR_BUDGET,
    )


sentence_window_answer = answer_from_context_fn(sentence_window_context)
sentence_window_df = run_shared_eval(sentence_window_answer, qna_dict)
sentence_window_df


### 🔍 结果透视：Sentence Window 增强了什么？

检索为**句级向量 top-k**（本节 `k=4`）。对每个命中句按 `sentence_id` 在 `neighbor_map` 中取前后各 `WINDOW_SIZE` 句（本节为 **2**），合并去重后得到扩展文本，再经 **`CONTEXT_CHAR_BUDGET`** 截断后送入 LLM（见下方 inspect 末尾预览）。

In [ ]:
def inspect_sentence_window(question):
    hit_ids = sentence_window_hit_ids(question)
    print(f"❓ 问题: {question}\n")
    for i, hid in enumerate(hit_ids):
        print(f"--- 命中句 {i+1} ---\n")
        print(f"[原始命中]: {sentence_map[hid]}")
        wids = sorted(neighbor_map.get(hid, [hid]))
        window_text = "\n".join(sentence_map[j] for j in wids)
        print(f"[增强窗口]:\n{window_text}\n")
    print("=" * 50)
    # 与 sentence_window_context / sentence_window_answer 使用同一条上下文
    print_llm_context_preview(
        sentence_window_context(question),
        "sentence_window_answer",
        "纯句窗",
    )

test_q = list(qna_dict.keys())[0]
inspect_sentence_window(test_q)

### Sentence Window 结果分析

（文末**同题对比表**与提分题列表怎么读，见 **「本节通用读法」**。）

- **本方法看点**：句窗相对 Baseline 的提分题；算法/模型、F1、宏微平均等常体现「邻句补全」。
- **局限**：窗口固定；证据分散在不相邻段落时需段落级方案（如 Small-to-Big）。


## Small-to-Big（父文档检索）

Sentence Window 能补回前后几句，但如果证据散落在段落的不同位置呢？比如段落开头有概念定义，中间有具体例子，结尾有对比总结——这些信息不在某一句的邻域里，而是分散在整个段落中。

> **运行顺序**：本单元会建立子块向量库；**AutoMerging** 下一节复用这里的 `child_retriever`、`child_to_parent`、`parent_texts` 等。生成用上下文 = **由子块命中映射出的父块文本**（去重后最多 3 个父块拼接），再按 **`CONTEXT_CHAR_BUDGET`** 截断。

### 核心思想

这就引出了一个经典的切块两难：**小块召回准但信息碎，大块信息全但相似度容易偏移**。Small-to-Big 用一个简单的分层策略同时解决这两个问题：

1. **索引时**：创建两级分块
   - **子块**：小块用于精确检索，嵌入向量存储在向量库
   - **父块**：大块用于提供完整上下文，存储在文档库
   - 记录每个子块属于哪个父块（`child_id → parent_id`）

2. **检索时**：
   - 用子块做精确召回定位
   - 不直接返回子块内容，而是通过映射找到它所属的父块
   - 把整个父块送给 LLM

### 与 Sentence Window 的区别

| 维度 | Sentence Window | Small-to-Big |
|---|---|---|
| 检索粒度 | 句子 | 子块（可自定义大小） |
| 上下文来源 | 固定窗口（前后 N 句） | 父块（语义完整的段落） |
| 灵活性 | 窗口大小固定 | 父块大小可调 |
| 适用场景 | 连续叙述文本 | 有清晰段落结构的文档 |

这样检索端享受小块的精度优势，生成端享受大块的完整性优势。适合有清晰段落结构的文档（教材、技术文档、论文）。局限在于：如果文档结构极不规则（比如对话记录、日志），父子映射本身就不可靠。

In [ ]:
documents = list(get_cleaned_pdf_documents())
# 子块写入向量库做检索，父块仅存内存；下一节 AutoMerging 复用 child_to_parent 等结构

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=480,
    chunk_overlap=60,
    separators=["\n\n", "\n", "。", "；", "：", " ", ""],
    keep_separator=True,
)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", "。", "；", "：", " ", ""],
    keep_separator=True,
)

parent_docs = parent_splitter.split_documents(documents)
parent_texts = [d.page_content for d in parent_docs]
print(f"父块数量: {len(parent_texts)}")

child_texts = []
child_to_parent = {}
for p_idx, p_doc in enumerate(parent_docs):
    for c in child_splitter.split_text(p_doc.page_content):
        c = c.strip()
        if not c:
            continue
        c_idx = len(child_texts)
        child_texts.append(c)
        child_to_parent[c_idx] = p_idx

print(f"子块数量: {len(child_texts)}")
print(f"平均每个父块的子块数: {len(child_texts) / max(1, len(parent_texts)):.1f}")

persist_dir_child = "./chroma_db/small_to_big"
child_docs = [
    Document(
        page_content=txt,
        metadata={"child_id": str(i), "parent_id": str(child_to_parent[i])},
    )
    for i, txt in enumerate(child_texts)
]
child_vs = open_or_build_chroma(
    persist_dir_child,
    child_docs,
    [f"child-{i}" for i in range(len(child_docs))],
)
child_retriever = child_vs.as_retriever(search_kwargs={"k": 4})


def small_to_big_hit_ids(question: str):
    return hit_child_ids_from_docs(retriever_hits(child_retriever, question))


def small_to_big_parent_ids(hit_ids):
    return sorted({child_to_parent[i] for i in hit_ids})


def small_to_big_expanded_join(parent_ids) -> str:
    # 去重后的父块按 id 序，最多拼接 3 段（未做 CONTEXT_CHAR_BUDGET）
    return "\n\n".join(parent_texts[i] for i in parent_ids[:3])


def small_to_big_context(question: str) -> str:
    return trim_context_to_budget(
        small_to_big_expanded_join(small_to_big_parent_ids(small_to_big_hit_ids(question))),
        CONTEXT_CHAR_BUDGET,
    )


small_to_big_answer = answer_from_context_fn(small_to_big_context)
small_to_big_df = run_shared_eval(small_to_big_answer, qna_dict)
small_to_big_df


### 🔍 结果透视：Small-to-Big 如何找回父文档？

Small-to-Big 的关键在于**检索子块，返回父块**。我们看看具体的子块命中情况以及对应的父块是否更完整：

In [ ]:
def inspect_small_to_big(question):
    hit_ids = small_to_big_hit_ids(question)

    print(f"❓ 问题: {question}\n")
    for i, cid in enumerate(hit_ids):
        pid = child_to_parent[cid]
        print(f"--- 子块命中 {i+1} ---\n")
        print(f"[子块内容]: {child_texts[cid]}")
        print(f"[对应父块]:\n{parent_texts[pid]}\n")
    print("=" * 50)
    # 与 small_to_big_context / small_to_big_answer 使用同一条上下文
    print_llm_context_preview(small_to_big_context(question), "small_to_big_answer", "纯父块")

inspect_small_to_big(test_q)

### Small-to-Big 结果分析

父/子块用 LangChain `RecursiveCharacterTextSplitter` 在按页清洗后的 `Document` 上切分；**子块检索 → 父块回填**的映射仍为本节手写逻辑。

（**提分 / 回退 / 持平**与平均分见 **「本节通用读法」**；F1、宏微平均等题可重点看父块是否比 Baseline 定长 chunk 更完整。）

- **局限**：切分仍依赖 `chunk_size` 与分隔符；结构混乱时父块未必是语义段落。


## AutoMerging（自动合并检索）

Small-to-Big 里有一个隐含假设：每个 child 命中后，直接回填它所属的 parent 就行。但实际场景中经常出现这样的情况——**同一个 parent 下有好几个 child 都被命中了**。这时候如果按 Small-to-Big 的逻辑，同一个 parent 会被重复回填，既浪费 token 又可能引入冗余。

> **依赖**：须先运行 **Small-to-Big** 索引单元（同一 `child_retriever` 与层级结构）。最终送入 LLM 的文本 = **仅 AutoMerge 拼出的扩展**（整父块或稀疏子块片段），再按 **`CONTEXT_CHAR_BUDGET`** 截断。

### 核心思想

AutoMerging 的思路是：与其被动回填，不如主动判断——**如果一个 parent 下被命中的 child 比例超过了阈值，就直接合并整个 parent 块作为上下文**。

具体流程：
1. **索引时**：和 Small-to-Big 一样，创建层级文档结构
2. **检索时**：
   - 先用叶子块（最小粒度块）做检索
   - 统计每个 parent 下有多少叶子被命中，算一个命中密度（命中数 / 总叶子数）
   - 密度超过阈值（`MERGE_THRESHOLD`，默认 0.30）**或**同一 parent 下命中子块数 ≥ `MIN_CHILD_HITS_FOR_MERGE`（默认 2）时，整体提升该 parent
   - 否则仅拼接被命中的子块文本（单证据且密度低）

### 与 Small-to-Big 的区别

| 维度 | Small-to-Big | AutoMerging |
|---|---|---|
| 返回策略 | 每个命中子块映射父块（本实现按 `parent_id` 去重） | 根据命中密度决定是否抬升整父块 |
| 重复问题 | 多子块同父时只保留一份父块文本 | 合并逻辑避免同一父块重复拼接 |
| 噪声控制 | 无 | 通过阈值过滤低相关性父块 |
| 复杂度 | 低 | 中（需要计算命中密度） |

这个阈值是 AutoMerging 的核心调控参数：设得太低，几乎所有 parent 都会被合并，噪声增多；设得太高，行为退化成 Small-to-Big。适合文档有明显层级结构、且证据经常分散在同一 parent 内多个位置的场景。

In [ ]:
# 依赖上一格 Small-to-Big：child_to_parent、child_retriever、parent_texts、child_texts
leaf_per_parent = {}
for c_idx, p_idx in child_to_parent.items():
    leaf_per_parent.setdefault(p_idx, []).append(c_idx)

print(f"父块数量: {len(leaf_per_parent)}")
print(f"平均每个父块的子块数: {sum(len(v) for v in leaf_per_parent.values()) / len(leaf_per_parent):.1f}")

MERGE_THRESHOLD = 0.30
MIN_CHILD_HITS_FOR_MERGE = 2  # 同父多子块命中 → 抬整父块（分散证据），避免只剩碎片而弱于 STB


def auto_merge_hit_ids(question: str):
    return set(hit_child_ids_from_docs(retriever_hits(child_retriever, question)))


def auto_merge_parent_stats(hit_ids, merge_threshold: float = MERGE_THRESHOLD):
    stats = []
    for p_idx, leaves in leaf_per_parent.items():
        hit_count = sum(1 for lid in leaves if lid in hit_ids)
        if hit_count == 0:
            continue
        total = len(leaves)
        ratio = hit_count / max(1, total)
        stats.append(
            {
                "parent_id": p_idx,
                "hit_count": hit_count,
                "total_children": total,
                "ratio": ratio,
                "is_merged": ratio >= merge_threshold or hit_count >= MIN_CHILD_HITS_FOR_MERGE,
            }
        )
    return stats


def auto_merge_extend_from_hit_ids(hit_ids, merge_threshold: float = MERGE_THRESHOLD):
    """仅构造 AutoMerge 侧的扩展文本，以及合并父块数 / 稀疏子块数（供 inspect 复用）。"""
    merged_pidx = []
    sparse_parts = []
    seen_parent = set()
    for item in auto_merge_parent_stats(hit_ids, merge_threshold):
        p_idx = item["parent_id"]
        if item["is_merged"]:
            if p_idx not in seen_parent:
                merged_pidx.append(p_idx)
                seen_parent.add(p_idx)
        else:
            for lid in leaf_per_parent[p_idx]:
                if lid in hit_ids:
                    sparse_parts.append(child_texts[lid])
    merged_pidx.sort(key=lambda p: min(leaf_per_parent[p]))
    merged_parents = [parent_texts[p] for p in merged_pidx]
    # 整父块优先，再缀少量「未抬升」的命中子块片段；段数封顶以控制噪声与长度
    max_context_parts = 5
    max_sparse_slots = 6
    parts = merged_parents + sparse_parts[:max_sparse_slots]
    ext = "\n\n".join(parts[:max_context_parts])
    return ext, len(merged_parents), min(len(sparse_parts), max_sparse_slots)


def auto_merge_context(question: str, merge_threshold: float = MERGE_THRESHOLD, verbose: bool = True) -> str:
    hit_ids = auto_merge_hit_ids(question)
    ext, n_par, n_sp = auto_merge_extend_from_hit_ids(hit_ids, merge_threshold)
    if verbose:
        print(f"  -> 合并父块数: {n_par}，稀疏子块片段: {n_sp}")
    return trim_context_to_budget(ext, CONTEXT_CHAR_BUDGET)


def auto_merge_eval_context(question: str) -> str:
    return auto_merge_context(question, verbose=True)


auto_merge_answer = answer_from_context_fn(auto_merge_eval_context)
auto_merging_df = run_shared_eval(auto_merge_answer, qna_dict)
auto_merging_df


### 🔍 结果透视：AutoMerging 如何实现自动合并？

AutoMerging 不只是简单的回填，它会计算**命中密度**。我们看看在同一个问题下，哪些父块被触发了合并逻辑；**代码单元末尾**会打印与 `auto_merge_answer` 一致的「纯 AutoMerge 扩展」送入 LLM 预览。

In [ ]:
def inspect_auto_merging(question, threshold=None):
    thr = MERGE_THRESHOLD if threshold is None else threshold
    hit_ids = auto_merge_hit_ids(question)
    stats = auto_merge_parent_stats(hit_ids, thr)

    print(f"❓ 问题: {question}\n")
    for item in sorted(stats, key=lambda x: x["parent_id"]):
        pid = item["parent_id"]
        count = item["hit_count"]
        total_children = item["total_children"]
        ratio = item["ratio"]

        print(f"--- 父块 {pid} (命中 {count}/{total_children}, 比例 {ratio:.1%}) ---\n")
        if item["is_merged"]:
            print(f"✅ [已合并]: 使用完整父块内容\n{parent_texts[pid][:200]}...")
        else:
            print(f"❌ [未合并]: 仅使用命中的子块片段")
    print("=" * 50)
    # 与 auto_merge_context / auto_merge_answer 使用同一条上下文
    print_llm_context_preview(
        auto_merge_context(question, thr, verbose=False),
        "auto_merge_answer",
        "纯 AutoMerge",
    )


inspect_auto_merging(test_q)



### AutoMerging 结果分析

当某父块满足 **命中子块比例 ≥ MERGE_THRESHOLD**，或 **同一父块下命中子块数 ≥ MIN_CHILD_HITS_FOR_MERGE**（分散多证据）时抬升整父块；否则只用被命中的子块片段。

（对比表读法见 **「本节通用读法」**；可与 **Small-to-Big**（始终抬父块、最多 3 个）对照看阈值带来的「整段 vs 碎片」权衡。）

- **局限**：依赖层级切分；阈值需调参。


## 同题对比总览（共用测试集）

下表将 Baseline 与各增强在同一题上的 **0～2 分**并列。**严格提分**：该方法得分 **严格大于** Baseline 该题得分。阅读 inspect、公平对比与提分/回退列表的说明见上文 **「本节通用读法」**。

若配置了 **WIN_TARGETS**，可逐项核对该题是否相对 Baseline 提分。

本格运行结束会写出 **`context_enhance_compare_scores.csv`**（仅 `train_idx` + 四列 0～2 分，便于脚本读入）、**`context_enhance_compare_full.json`**（含 `question` 等完整列）、**`context_enhance_eval_summary.json`**（均值、Δ、前缀子集）。


对比表也是 6.1 首次出现的概念：把多个方法的 `run_shared_eval` 结果按 `question` 列对齐为一张同题对比表。先在本节完整 inline 定义一次，再放进 `_common.py`，6.2 / 6.3 节直接 `from _common import build_compare_table` 复用。


In [ ]:
def build_compare_table(dfs: list[pd.DataFrame], names: list[str]) -> pd.DataFrame:
    """把多个 run_shared_eval 的结果按 question 列对齐拼接为同题对比表。"""
    if len(dfs) != len(names):
        raise ValueError("dfs 与 names 长度必须一致")
    base = dfs[0][["question"]].copy()
    base[names[0]] = dfs[0]["rag_eval_results"].values
    out = base
    for df, name in zip(dfs[1:], names[1:]):
        out = out.merge(
            df[["question", "rag_eval_results"]].rename(columns={"rag_eval_results": name}),
            on="question",
            how="outer",
        )
    return out.reset_index(drop=True)

compare_df = build_compare_table(
    [baseline_df, sentence_window_df, small_to_big_df, auto_merging_df],
    names=["baseline", "sentence_window", "small_to_big", "auto_merging"],
)
compare_df

In [ ]:
# 分项分 CSV：question + 四列分（避免长答案触发 csv 解析问题）
_score_csv_cols = ["question", "baseline", "sentence_window", "small_to_big", "auto_merging"]
compare_df[_score_csv_cols].to_csv("context_enhance_compare_scores.csv", index=False, encoding="utf-8")
# 完整行（含 question）用 JSON，由 pandas 转义换行与引号
compare_df.to_json("context_enhance_compare_full.json", orient="records", force_ascii=False, indent=2)
print("✅ 已写 context_enhance_compare_scores.csv（分项）；context_enhance_compare_full.json（含题干等）")


def summarize_context_eval(compare_df, prefix_ks=(10, 20, 30, 50, 100), json_path="context_enhance_eval_summary.json"):
    """前缀子集均值 + 结构化摘要 JSON（不重评，仅聚合 compare_df）。
    顺序遵循 compare_df 的行序（即 QA_INDICES 顺序）。
    """
    import json as _json

    df = compare_df.reset_index(drop=True)
    cols = [
        ("Baseline", "baseline"),
        ("Sentence Window", "sentence_window"),
        ("Small-to-Big", "small_to_big"),
        ("AutoMerging", "auto_merging"),
    ]
    n = len(df)
    max_pts = 2 * n
    b_mean = float(df["baseline"].mean())
    summary = {
        "n_questions": n,
        "max_score_sum": max_pts,
        "baseline_mean": b_mean,
        "baseline_sum": int(df["baseline"].sum()),
        "methods": {},
        "prefix_subsets": {},
    }
    for name, col in cols[1:]:
        mu = float(df[col].mean())
        summary["methods"][col] = {
            "mean": mu,
            "sum": int(df[col].sum()),
            "delta_vs_baseline": round(mu - b_mean, 6),
            "strict_wins_vs_baseline": int((df[col] > df["baseline"]).sum()),
        }
    print("\n--- 前缀子集（按 QA_INDICES 顺序的前 K 行，不重评）---\n")
    for k in prefix_ks:
        m = min(k, n)
        sub = df.iloc[:m]
        mp = 2 * m
        bm = float(sub["baseline"].mean())
        ps = {"n": m, "max_pts": mp, "baseline_mean": bm, "baseline_sum": int(sub["baseline"].sum()), "methods": {}}
        print(f"前 {k} 题 → n={m}，满分 {mp}")
        print(f"  Baseline:   {bm:.3f}（总分 {int(sub['baseline'].sum())}/{mp}）")
        for name, col in cols[1:]:
            smu = float(sub[col].mean())
            ss = int(sub[col].sum())
            ps["methods"][col] = {
                "mean": smu,
                "sum": ss,
                "delta_vs_baseline": round(smu - bm, 6),
            }
            print(f"  {name}: {smu:.3f}（总分 {ss}/{mp}），Δ={smu - bm:+.3f}")
        print()
        summary["prefix_subsets"][str(k)] = ps
    with open(json_path, "w", encoding="utf-8") as f:
        _json.dump(summary, f, ensure_ascii=False, indent=2)
    print(f"✅ 评估摘要 JSON：{json_path}")


_cols = [("Sentence Window", "sentence_window"), ("Small-to-Big", "small_to_big"), ("AutoMerging", "auto_merging")]
_n = len(compare_df)
_max_pts = 2 * _n

print("\n--- 各路径平均分（rag_eval_results 为 0～2 分）---")
_b_mean = compare_df["baseline"].mean()
print(f"  Baseline:   {_b_mean:.3f}（总分 {int(compare_df['baseline'].sum())}/{_max_pts}）")
for name, col in _cols:
    mu = compare_df[col].mean()
    print(f"  {name}: {mu:.3f}（总分 {int(compare_df[col].sum())}/{_max_pts}），相对 Baseline Δ={mu - _b_mean:+.3f}")

print("\n--- 全表统计：该方法得分 高于 Baseline 的题数（严格提分）---")
for name, col in _cols:
    n_up = int((compare_df[col] > compare_df["baseline"]).sum())
    print(f"  {name}: {n_up}/{_n}")

print("\n--- 本跑「提分题」行号（得分严格大于 Baseline）---")
for name, col in _cols:
    xs = compare_df.index[compare_df[col] > compare_df["baseline"]].tolist()
    print(f"  {name}: {xs}")

print("\n--- 回退：Baseline 得分高于该方法 ---")
for name, col in _cols:
    bad = compare_df.index[compare_df["baseline"] > compare_df[col]].tolist()
    print(f"  {name}: {len(bad)}/{_n} 题 -> {bad}")

print("\n--- 持平：与 Baseline 同分 ---")
for name, col in _cols:
    eq = compare_df.index[compare_df["baseline"] == compare_df[col]].tolist()
    print(f"  {name}: {len(eq)}/{_n} 题 -> {eq}")

summarize_context_eval(compare_df)

### 前缀子集与 JSON 摘要（不重评）

**上一格「同题对比」末尾**已调用 `summarize_context_eval`：打印按 `train_idx` 升序的前 K 题（10/20/30/50/100）均值，并写出 **`context_enhance_eval_summary.json`**（便于脚本或 CI 读取）。若仅想从 CSV 离线重算，可再运行下一格或终端 `python3 report_context_prefix.py`。

In [ ]:
# 可选：仅打开 CSV 重打前缀表（新内核未跑对比格时）
from pathlib import Path
import pandas as pd

_csv = Path("context_enhance_compare_scores.csv")
if not _csv.is_file():
    print("无 context_enhance_compare_scores.csv，跳过（请先运行「同题对比」格）。")
else:
    _df = pd.read_csv(_csv).reset_index(drop=True)

    def _prefix_only(df, ks=(10, 20, 30, 50, 100)):
        cols = [
            ("Baseline", "baseline"),
            ("Sentence Window", "sentence_window"),
            ("Small-to-Big", "small_to_big"),
            ("AutoMerging", "auto_merging"),
        ]
        n_all = len(df)
        print("--- 自 CSV 前缀子集（与对比格 summarize_context_eval 一致）---\n")
        for k in ks:
            n = min(k, n_all)
            sub = df.iloc[:n]
            mp = 2 * n
            bm = float(sub["baseline"].mean())
            print(f"前 {k} 题 → n={n}，满分 {mp}")
            print(f"  Baseline:   {bm:.3f}（总分 {int(sub['baseline'].sum())}/{mp}）")
            for name, col in cols[1:]:
                mu = float(sub[col].mean())
                print(f"  {name}: {mu:.3f}（总分 {int(sub[col].sum())}/{mp}），Δ={mu - bm:+.3f}")
            print()

    _prefix_only(_df)

## 实验结果汇总

**阅读顺序**：先看上表 **同题对比**与打印的**平均分 / 提分题**，再看各 `*_df`。裁判为 **0～2 分**，宜结合要点完整度理解，不必以「全 2 分」为唯一结论；若需调参可改 `CONTEXT_CHAR_BUDGET`、窗口大小或父块数量（改后请删对应 `chroma_db` 子目录重建）。

| train 下标 | 题意摘要 | 备注 |
|---|---|---|
| 0 | 算法 / 模型 / 关系 | Small-to-Big 示范题 |
| 1 | 泛化 + 西瓜例子 | Sentence Window 示范题（长答） |
| 2 | 奥卡姆与房价模型 | AutoMerging 示范题 |
| 3 | 测试集分布变化下算法比较 | 论述题 |
| 4 | 为何评估模型；经验 vs 泛化误差 | 第2章衔接 |
| 5 | 交叉验证 vs 单次留出 | 邻句可补全定义 |
| 6 | F1 调和平均与 β | 短答 |
| 7 | 宏平均 vs 微平均 | 中长篇 |
| 26 | 式(4.8) λ | 决策树连续属性 |
| 27 | 图4-2 四次划分 | 好瓜/坏瓜边界 |

### 定性结论

- **Baseline**：常规分块 + top-k；叙述型文档上仍可能出现上下文碎片。
- **Sentence Window**：轻量；注意目录/长段（本节已切段缓解）。
- **Small-to-Big**：小块定位 + 父块生成。
- **AutoMerging**：阈值在整父块与叶子片段间折中。


## 方法特性对比

| 方法 | 核心思想 | 典型修复问题 | 新增复杂度 | 最适合文档形态 |
|---|---|---|---|---|
| baseline | 无（作为对照） | 无 | 低 | 任意 |
| Sentence Window | 命中句子后扩展前后窗口 | 命中句缺邻域证据 | 低 | 连续叙述文本 |
| Small-to-Big | 子块检索，父块生成 | 小块准但不完整 | 中 | 段落层级清晰 |
| AutoMerging | 根据命中密度合并父块 | 多子块分散命中同一父块 | 中-高 | 树状层级明显 |
| Late Chunking（理论） | 索引时让 chunk 看到完整文档 | embedding 缺少文档全局信息 | 低（但模型受限） | 需长上下文 embedding 模型 |

## 前沿方法：Late Chunking（理论介绍）

### 传统流程的问题
传统 RAG 的流程是先切分，再分别嵌入——每个 chunk 独立通过 embedding 模型，丢失了跨 chunk 的语义关联。

### Late Chunking 思路
Late Chunking 颠倒了这个顺序：
1. 先将**整个文档**送入长上下文 embedding 模型（如 jina-embeddings-v2），获得每个 token 的上下文化表示
2. 再按预设边界切分 token embeddings，对每个 chunk 的 token embeddings 做池化得到 chunk embedding

这样每个 chunk 的向量都见过完整文档上下文，天然缓解了上下文割裂问题。

### 与本章方法的对比

| 维度 | Sentence Window / Small-to-Big / AutoMerging | Late Chunking |
|---|---|---|
| 增强时机 | 检索时（命中后恢复邻域） | 索引时（embedding 阶段） |
| 额外存储 | 需要维护邻居映射 / 父子关系 | 不需要 |
| 模型依赖 | 无特殊要求 | 需要长上下文 embedding 模型 |
| 实现复杂度 | 中 | 低（但模型选择受限） |

### 局限
- 依赖支持长上下文的 embedding 模型（如 jina-embeddings-v2、nomic-embed），智谱 embedding-3 等通用 embedding 模型不直接支持此模式
- 文档超过模型上下文窗口时需要分段处理
- 目前 LangChain 生态无开箱即用的 Late Chunking 组件

### 本节为什么不做代码示例
Late Chunking 需要特殊的 embedding 模型和自定义 tokenizer 操作，与本节统一使用本地 bge-small-zh-v1.5 的可复现实验设定不兼容。此处仅作概念介绍，帮助读者建立还有一类索引时上下文增强的认知。

## 如何选择

- 如果问题经常只差前后两句：先用 Sentence Window。
- 如果文档天然有章节层级：优先 Small-to-Big。
- 如果证据常分散在同一父块多个子块：优先 AutoMerging。
- 若问题本质是多步推理或多轮交互，应转到流程增强或系统增强。
- 如果问题出在 chunk 本身缺少文档语境（脱离上下文后语义不完整），这属于**索引阶段**优化，应回到第 3 章的 CCH / Contextual Retrieval。本章的上下文增强解决的是检索后恢复邻域，而非索引时缺少语境。

## 下一步

如果你发现问题的根源不是上下文不全，而是一次检索流程本身不够——比如需要多步推理、需要先评估检索质量再决定下一步——请继续学习 `2. 流程增强.ipynb`。

### 学习检查点

- 你能区分检索相关但上下文不足与流程不足吗？
- 你能解释 Sentence Window 与 Small-to-Big 的关键差异吗？
- 你知道 AutoMerging 的阈值会如何影响召回上下文长度吗？
- 你能说明为何在统一 `CONTEXT_CHAR_BUDGET` 下，增强路径仍可能相对 Baseline **平均分变低或变高**吗（检索对象不同、截断位置不同）？